# Install Dependecies

In [11]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [12]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [13]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import re
import time
from collections import Counter
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer

# Import datasets

In [5]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [6]:
# tokeniser
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [7]:
LANGUAGES = ["ar", "ko", "te"]

In [8]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# 3 Week 37: Structured Span Prediction
Convert the character-level answer offsets into BIO labels over context tokens. Add automatic checks for at least the following cases: an answer at character 0, a multi-token answer, punctuation adjacent to an answer and an unanswerable example. Document how subword pieces are handled if applicable. Implement one question-conditioned sequence labeller for the group: the representation of the question must influence the predicted label for every context token. Compare it with a simple span baseline. An empty-output baseline is sufficient. If you use lexical overlap, select context tokens using only overlap with the question (optionally after fixed preprocessing or translation), convert the best contiguous run to a span and never use gold answer text or offsets. The correct output for an unanswerable question is an empty span. Evaluate and analyse the models according to Section 1.


In [14]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

In [15]:
def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets